# Constrained Optimization

| Difficulty | ██████░░░░ 6/10 |
| :--- | :--- |
| Prerequisites | 04.04 |
| Time | ~75 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/04_Optimization/05_constrained_optimization.ipynb)

---

## 🎯 Learning Objective

Learn Lagrange multipliers and KKT conditions for solving optimization problems subject to equality and inequality constraints.

---

## 📐 Theory

Every optimization problem so far in this module has been **unconstrained**: minimize $f(\mathbf{w})$
over all of $\mathbb{R}^n$. Many real problems restrict $\mathbf{w}$ to a feasible region —
support vector machines, for instance, need the margin constraint satisfied by every data
point, not just a low loss. This notebook covers the machinery for that: Lagrange multipliers
and the KKT conditions.

### Equality constraints: Lagrange multipliers

To minimize $f(\mathbf{w})$ subject to $g(\mathbf{w})=0$, form the **Lagrangian**:

$$\mathcal{L}(\mathbf{w}, \lambda) = f(\mathbf{w}) - \lambda\, g(\mathbf{w})$$

At a constrained optimum, $\nabla f$ and $\nabla g$ must be *parallel* — if they weren't, you
could slide along the constraint surface $g=0$ and still decrease $f$, contradicting
optimality. "Parallel gradients" is exactly what $\nabla_{\mathbf{w}}\mathcal{L}=\mathbf{0}$
enforces: $\nabla f(\mathbf{w}) = \lambda \nabla g(\mathbf{w})$. Together with the original
constraint $\nabla_\lambda \mathcal{L} = -g(\mathbf{w}) = 0$, this gives a system of equations
that pins down both $\mathbf{w}^*$ and the multiplier $\lambda$ (whose value, as a bonus, tells
you the optimum's sensitivity to relaxing the constraint — recall the sensitivity reading of a
derivative from `02.02`).

### Inequality constraints: KKT conditions

For $g(\mathbf{w}) \le 0$, the constraint might be **active** (binding, $g(\mathbf{w}^*)=0$,
pressing against the boundary) or **inactive** ($g(\mathbf{w}^*)<0$, the unconstrained optimum
already satisfies it). The **Karush-Kuhn-Tucker (KKT) conditions** generalize Lagrange
multipliers to handle both cases at once, using a multiplier $\mu \ge 0$:

$$\nabla f(\mathbf{w}^*) + \mu \nabla g(\mathbf{w}^*) = \mathbf{0}, \qquad
\mu \ge 0, \qquad g(\mathbf{w}^*) \le 0, \qquad \mu\, g(\mathbf{w}^*) = 0$$

The last equation is **complementary slackness**: either the constraint is active
($g(\mathbf{w}^*)=0$) or its multiplier is zero ($\mu=0$) — never both nonzero. Intuitively, a
constraint only "pushes back" on the optimum (nonzero $\mu$) when you're pressed right up
against it; if the unconstrained optimum was already feasible, the constraint does nothing and
its multiplier vanishes.

### The dual problem

Define the **dual function** $q(\mu) = \min_{\mathbf{w}} \mathcal{L}(\mathbf{w}, \mu)$ — the
best the Lagrangian can do for a fixed multiplier. Maximizing $q$ over $\mu \ge 0$ (the **dual
problem**) gives a lower bound on the original ("primal") problem's optimal value —
$q(\mu)\le p^*$ for every feasible $\mu\ge0$, called **weak duality**, and it holds
unconditionally, convex or not. **Strong duality** (the bound is *tight*, so the dual's optimal
value $d^*$ equals the primal's $p^*$ exactly) is a strictly stronger claim: convexity of the
primal problem alone does not guarantee it. The standard sufficient condition is **Slater's
condition** — there exists a strictly feasible point ($g(\mathbf{w})<0$ for every inequality
constraint, not merely $\le 0$) — which SVMs satisfy comfortably, which is why the SVM dual is
safe to solve in their place. Convex problems that violate every constraint qualification can
have a nonzero duality gap: Boyd & Vandenberghe (2004, §5.2.3) give
$\min_{x,y} e^{-x}$ subject to $x^2/y \le 0$, $y>0$ as a convex problem with $p^*=1$ but
$d^*=0$, precisely because no strictly feasible point exists. Solving the dual is sometimes far
easier when strong duality *does* hold — SVMs are the canonical example, where the dual turns
into a problem expressed purely via dot products between data points, which is what makes the
kernel trick possible (previewed here, covered fully in `11.02`).

### Penalty methods

An alternative to solving the Lagrangian system directly: convert a constrained problem into an
*unconstrained* one by adding a penalty term that punishes constraint violation,
$f(\mathbf{w}) + \rho \max(0, g(\mathbf{w}))^2$ for growing $\rho$. As $\rho\to\infty$, the
penalized minimizer converges to the constrained optimum. This is crude compared to solving KKT
directly, but it reduces a constrained problem to something `04.02`'s gradient descent already
knows how to handle — a pattern that reappears whenever regularization is described as a "soft
constraint" on model complexity.


### Projected gradient descent

Penalty methods convert a constrained problem into an unconstrained one; **projected gradient
descent** keeps the constraint explicit instead. At each step, take a normal gradient step and
then *project* back onto the feasible set $C$:

$$w_{t+1} = \text{Proj}_C\!\big(w_t - \eta\,\nabla f(w_t)\big)$$

where the **Euclidean projection** onto a closed convex set $C$ is:

$$\text{Proj}_C(v) \;=\; \arg\min_{w \in C}\;\|w - v\|_2^2$$

— the nearest feasible point. For convex $C$ and convex $L$-smooth $f$, projected GD converges
at $O(1/T)$ — the same rate as unconstrained GD (`04.03`), because projection onto a convex set
is a *non-expansive* operation (it never increases distances).

The algorithm is practical only when the projection itself is cheap. Three cases cover the vast
majority of ML applications:

| Constraint set $C$ | Projection | Cost |
| :--- | :--- | :--- |
| **Box** $l_i \le w_i \le u_i$ | Clip each coordinate: $[\text{Proj}_C(v)]_i = \text{clip}(v_i, l_i, u_i)$ | $O(d)$ |
| **Probability simplex** $\{w \ge 0,\; \sum w_i = 1\}$ | Sort-and-threshold (Duchi et al., 2008): sort $v$ descending, find $\rho = \max\{j : v_{(j)} - \tau > 0\}$ with $\tau = (\sum_{i=1}^{\rho} v_{(i)} - 1)/\rho$, then $[\text{Proj}(v)]_i = \max(v_i - \tau, 0)$ | $O(d \log d)$ |
| **$\ell_2$ ball** $\{w : \|w\| \le r\}$ | Rescale if needed: $\text{Proj}(v) = v \cdot \min(1,\; r/\|v\|)$ | $O(d)$ |

These cover non-negative matrix factorization (box), attention/portfolio weights (simplex), and
norm-constrained training (ball). When projection is *expensive* (e.g., nuclear-norm constraints),
Frank-Wolfe (`04.12`) offers a projection-free alternative.

### The general KKT system

The Theory section above treated a single inequality constraint. The standard form handles
$m$ inequality and $p$ equality constraints simultaneously:

$$\min_{\mathbf{w}}\; f(\mathbf{w}) \quad \text{subject to} \quad
g_i(\mathbf{w}) \le 0,\; i=1,\ldots,m; \qquad h_j(\mathbf{w}) = 0,\; j=1,\ldots,p$$

The **Lagrangian** is:

$$\mathcal{L}(\mathbf{w}, \boldsymbol{\mu}, \boldsymbol{\lambda})
\;=\; f(\mathbf{w})
\;+\; \sum_{i=1}^{m} \mu_i\, g_i(\mathbf{w})
\;+\; \sum_{j=1}^{p} \lambda_j\, h_j(\mathbf{w})$$

A point $\mathbf{w}^*$ with multipliers $\boldsymbol{\mu}^*$, $\boldsymbol{\lambda}^*$
satisfies the **KKT conditions** if *all four* hold:

| Condition | Statement | Meaning |
| :--- | :--- | :--- |
| **Stationarity** | $\nabla f(\mathbf{w}^*) + \sum_{i} \mu_i^* \nabla g_i(\mathbf{w}^*) + \sum_{j} \lambda_j^* \nabla h_j(\mathbf{w}^*) = \mathbf{0}$ | Gradient of the Lagrangian vanishes |
| **Primal feasibility** | $g_i(\mathbf{w}^*) \le 0$ for all $i$; $\;h_j(\mathbf{w}^*) = 0$ for all $j$ | The point is feasible |
| **Dual feasibility** | $\mu_i^* \ge 0$ for all $i$ | Inequality multipliers are non-negative |
| **Complementary slackness** | $\mu_i^*\, g_i(\mathbf{w}^*) = 0$ for each $i = 1,\ldots,m$ | Each inequality is either active or its multiplier is zero |

This is exactly the system that `scipy.optimize.minimize(method='SLSQP')` solves internally.
For convex problems satisfying a constraint qualification (e.g., Slater's condition), KKT
conditions are both *necessary and sufficient* for global optimality — the single-constraint
intuition ("either the constraint pushes back, or it doesn't") extends identically to every
constraint independently.


---

In [ ]:
# Setup
import numpy as np
import sympy as sp
from scipy.optimize import minimize
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

At a constrained optimum, the objective's contour line is tangent to the constraint curve —
their gradients point in the same (or exactly opposite) direction. Plotting both makes this
tangency, and hence why $\nabla f = \lambda \nabla g$, visible directly.

In [ ]:
# Minimize f(x,y)=x^2+y^2 subject to g(x,y)=x+y-1=0 -- visualize the tangency condition
f = lambda x, y: x**2 + y**2
x_opt, y_opt = 0.5, 0.5  # solved analytically in the Theory section: x=y=0.5, lambda=1

x = np.linspace(-1, 2, 300)
y = np.linspace(-1, 2, 300)
X, Y = np.meshgrid(x, y)
Z = f(X, Y)

fig, ax = plt.subplots(figsize=(7, 6.5))
ax.contour(X, Y, Z, levels=15, cmap="viridis", alpha=0.7)
ax.plot(x, 1 - x, color="#C44E52", lw=2.5, label="constraint: x+y=1")
ax.plot(x_opt, y_opt, 'ko', markersize=10, zorder=5, label="constrained optimum")

# Draw both gradients at the optimum -- they should be parallel (KKT: grad f = lambda * grad g)
grad_f = np.array([2 * x_opt, 2 * y_opt])
grad_g = np.array([1.0, 1.0])  # gradient of x+y-1 is always (1,1)
ax.annotate("", xy=(x_opt + grad_f[0]*0.3, y_opt + grad_f[1]*0.3), xytext=(x_opt, y_opt),
            arrowprops=dict(arrowstyle="->", color="#4C72B0", lw=2.5))
ax.annotate("", xy=(x_opt + grad_g[0]*0.3, y_opt + grad_g[1]*0.3), xytext=(x_opt, y_opt),
            arrowprops=dict(arrowstyle="->", color="#DD8452", lw=2, linestyle="--"))
ax.text(x_opt + 0.35, y_opt + 0.25, "grad f", color="#4C72B0", fontsize=10)
ax.text(x_opt + 0.35, y_opt + 0.42, "grad g", color="#DD8452", fontsize=10)
ax.set_xlim(-0.5, 1.5); ax.set_ylim(-0.5, 1.5)
ax.set_xlabel("x"); ax.set_ylabel("y")
ax.set_title("grad f is parallel to grad g at the constrained optimum")
ax.legend(loc="upper right", fontsize=9)
ax.set_aspect('equal')
plt.show()

print(f"grad f at optimum: {grad_f}, grad g: {grad_g}")
print(f"ratio grad_f / grad_g = {grad_f / grad_g}  -- constant ratio confirms parallel vectors (lambda=1)")

## 🐍 Implementation from Scratch

We solve one equality-constrained and one inequality-constrained problem two independent ways:
symbolically via the Lagrangian/KKT stationarity equations, and numerically via
`scipy.optimize.minimize` -- confirming they agree.

In [ ]:
# --- Equality-constrained: minimize x^2+y^2 subject to x+y=1 ---
x, y, lam = sp.symbols('x y lam', real=True)
L_eq = x**2 + y**2 - lam * (x + y - 1)
stationarity = [sp.diff(L_eq, x), sp.diff(L_eq, y), sp.diff(L_eq, lam)]
sol_eq = sp.solve(stationarity, [x, y, lam])
print("Lagrangian stationarity (sympy):", sol_eq)

res_eq = minimize(lambda v: v[0]**2 + v[1]**2, [0, 0], constraints=[{'type': 'eq', 'fun': lambda v: v[0] + v[1] - 1}])
print(f"scipy.optimize.minimize:          x={res_eq.x[0]:.4f}, y={res_eq.x[1]:.4f}")
print(f"Match: {np.isclose(float(sol_eq[x]), res_eq.x[0]) and np.isclose(float(sol_eq[y]), res_eq.x[1])}")

# --- Inequality-constrained: minimize (x-2)^2+(y-2)^2 subject to x+y<=2, x>=0, y>=0 ---
# The unconstrained minimum (2,2) violates x+y<=2, so this constraint should be ACTIVE
# (binding) at the true optimum -- KKT predicts a nonzero multiplier there.
def obj(v):
    return (v[0] - 2)**2 + (v[1] - 2)**2

res_ineq = minimize(obj, [0, 0], constraints=[{'type': 'ineq', 'fun': lambda v: 2 - v[0] - v[1]}],
                     bounds=[(0, None), (0, None)])
x_star, y_star = res_ineq.x
g_at_star = x_star + y_star - 2   # should be ~0: constraint is active
print(f"\nInequality-constrained optimum: x={x_star:.4f}, y={y_star:.4f}")
print(f"Constraint x+y-2 at optimum: {g_at_star:.6f}  (near 0 => ACTIVE, matching KKT's prediction")
print(f"since the unconstrained optimum (2,2) violates x+y<=2)")

# Verify complementary slackness by solving the KKT stationarity system directly with sympy:
# grad f + mu * grad g = 0, with g(x,y) = x+y-2 (active, so g=0 exactly)
mu = sp.symbols('mu', nonnegative=True)
kkt_eqs = [sp.diff((x - 2)**2 + (y - 2)**2, x) + mu * 1,
           sp.diff((x - 2)**2 + (y - 2)**2, y) + mu * 1,
           x + y - 2]
sol_kkt = sp.solve(kkt_eqs, [x, y, mu])
print(f"\nKKT system solved symbolically: x={float(sol_kkt[x]):.4f}, y={float(sol_kkt[y]):.4f}, mu={float(sol_kkt[mu]):.4f}")
print(f"mu > 0 confirms the constraint is active, consistent with complementary slackness (mu * g = 0, g = 0).")

# --- Strong duality is NOT automatic from convexity alone: Slater's condition matters ---
# Boyd & Vandenberghe (2004, Sec 5.2.3): minimize e^{-x} subject to x^2/y <= 0, y > 0.
# This is a CONVEX problem (e^{-x} convex, x^2/y convex on y>0), but the only feasible points
# have x=0 exactly (x^2/y<=0 with y>0 forces x^2<=0, i.e. x=0) -- so no STRICTLY feasible point
# (x^2/y < 0) exists anywhere, meaning Slater's condition fails here.
def L_slater(x, y, mu):
    return np.exp(-x) + mu * x**2 / y

p_star_slater = 1.0  # forced to x=0 on the only feasible ray, so p* = e^0 = 1

print("Dual function g(mu) = inf_{x, y>0} L(x,y,mu), probed along the ray x=t, y=t^3:")
for mu in [0.0, 1.0, 10.0, 100.0]:
    tail_vals = [L_slater(t, t**3, mu) for t in [10, 100, 1000, 10000]]
    print(f"  mu={mu:6.1f}: L(t, t^3, mu) as t grows -> {[f'{v:.2e}' for v in tail_vals]}  (-> 0)")

d_star_slater = 0.0  # e^{-t} -> 0 and mu*t^2/t^3 = mu/t -> 0 along this ray, for EVERY mu >= 0,
                     # so g(mu) = 0 for every mu and d* = sup_mu g(mu) = 0
print(f"\nPrimal optimal p* = {p_star_slater}   Dual optimal d* = {d_star_slater}")
print(f"Duality gap p*-d* = {p_star_slater - d_star_slater} > 0 on a CONVEX problem --")
print("strong duality fails here because no strictly feasible point exists (Slater's condition")
print("needs x^2/y < 0 strictly, impossible for y>0), confirming convexity alone does not imply")
print("strong duality -- a constraint qualification like Slater's condition is still required.")


In [ ]:
# --- Projected gradient descent: simplex projection + portfolio optimization ---

def project_simplex(v):
    """Project v onto the probability simplex {w >= 0, sum(w) = 1}.
    
    Algorithm: sort-and-threshold (Duchi et al., 2008).
    1. Sort v in descending order.
    2. Find rho = max{j : v_(j) - tau > 0} where tau = (sum_{i<=rho} v_(i) - 1) / rho.
    3. Set w_i = max(v_i - tau, 0).
    """
    n = len(v)
    u = np.sort(v)[::-1]                    # sort descending
    cumsum = np.cumsum(u)
    rho = np.where(u - (cumsum - 1) / np.arange(1, n + 1) > 0)[0][-1]
    tau = (cumsum[rho] - 1) / (rho + 1)
    return np.maximum(v - tau, 0)

# Verify: random vectors should project to valid probability distributions
rng_proj = np.random.default_rng(42)
for _ in range(5):
    v_test = rng_proj.normal(size=5)
    w_proj = project_simplex(v_test)
    assert np.allclose(w_proj.sum(), 1.0), "weights must sum to 1"
    assert np.all(w_proj >= -1e-12), "weights must be non-negative"
print("Simplex projection: all sanity checks passed (sum=1, w>=0)")

# --- Portfolio optimization: minimize variance s.t. weights on the simplex ---
# min  w^T Sigma w   s.t.  w >= 0, sum(w) = 1
# This is the minimum-variance portfolio -- a classic simplex-constrained QP.

n_assets = 5
rng_port = np.random.default_rng(7)
# Generate a random positive-definite covariance matrix
A = rng_port.normal(size=(n_assets, n_assets))
Sigma = A.T @ A / n_assets + 0.1 * np.eye(n_assets)  # ensure well-conditioned

def portfolio_variance(w):
    return w @ Sigma @ w

def grad_portfolio_variance(w):
    return 2 * Sigma @ w

# Projected gradient descent
eta = 0.01
T = 2000
w = np.ones(n_assets) / n_assets           # start at equal weights (feasible)
history = [portfolio_variance(w)]

for t in range(T):
    g = grad_portfolio_variance(w)
    w_unconstrained = w - eta * g            # gradient step
    w = project_simplex(w_unconstrained)     # project back onto simplex
    history.append(portfolio_variance(w))

print(f"\nMinimum-variance portfolio (projected GD, {T} steps):")
print(f"  Optimal weights: {np.round(w, 4)}")
print(f"  Portfolio variance: {portfolio_variance(w):.6f}")
print(f"  Weights sum to: {w.sum():.6f}  (should be 1.0)")
print(f"  All weights >= 0: {np.all(w >= -1e-10)}")

# Cross-check with scipy (SLSQP solves the general KKT system internally)
from scipy.optimize import minimize as sp_minimize
res = sp_minimize(portfolio_variance, np.ones(n_assets) / n_assets,
                  jac=grad_portfolio_variance,
                  constraints=[{'type': 'eq', 'fun': lambda w: w.sum() - 1}],
                  bounds=[(0, None)] * n_assets, method='SLSQP')
print(f"\nScipy SLSQP solution:  {np.round(res.x, 4)}")
print(f"Scipy variance:        {res.fun:.6f}")
print(f"Match: {np.allclose(w, res.x, atol=1e-3)}")

# Convergence plot
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(history, color='#4C72B0', lw=1.5)
ax.axhline(res.fun, color='#C44E52', ls='--', lw=1, label='scipy optimum')
ax.set_xlabel('iteration')
ax.set_ylabel('portfolio variance  $w^T \Sigma w$')
ax.set_title('Projected GD on the simplex: minimum-variance portfolio')
ax.legend()
ax.set_yscale('log')
plt.tight_layout()
plt.show()


## 🤖 Why This Matters for AI

A **Support Vector Machine** finds the maximum-margin separating hyperplane by solving a
constrained problem: minimize $\frac{1}{2}\|\mathbf{w}\|^2$ subject to
$y_i(\mathbf{w}\cdot\mathbf{x}_i + b) \ge 1$ for every training point $i$. This is exactly the
KKT setting above: most points satisfy the margin constraint comfortably (inactive, multiplier
$\alpha_i=0$), while the points sitting exactly on the margin boundary — the **support
vectors** — have the constraint active ($\alpha_i > 0$). Complementary slackness is the precise
mathematical statement of why an SVM's decision boundary depends on only a handful of points
instead of the entire dataset. Below, we fit `sklearn`'s SVM on a toy linearly-separable
dataset and confirm directly: points with nonzero dual coefficients (the $\alpha_i y_i$ in
`svm.dual_coef_`) are exactly `svm.support_vectors_`, and every other point's multiplier is
zero — complementary slackness, verified on a real (if small) model.

In [ ]:
from sklearn.svm import SVC

rng = np.random.default_rng(0)
X = np.vstack([rng.normal([-2, -2], 0.6, size=(20, 2)), rng.normal([2, 2], 0.6, size=(20, 2))])
y_labels = np.array([-1] * 20 + [1] * 20)

svm = SVC(kernel="linear", C=1000)  # large C: closely approximates the hard-margin SVM above
svm.fit(X, y_labels)

n_support = len(svm.support_)
n_total = len(X)
print(f"Total training points: {n_total}")
print(f"Support vectors (active constraint, alpha_i > 0): {n_support}")
print(f"Points with alpha_i = 0 (constraint inactive, margin satisfied with room to spare): {n_total - n_support}")
print(f"\nThe decision boundary w={np.round(svm.coef_[0], 3)}, b={svm.intercept_[0]:.3f} depends")
print(f"ONLY on the {n_support} support vectors -- removing any non-support-vector point and")
print(f"re-fitting leaves the boundary UNCHANGED, which we verify next.")

# Verify: refit after removing a non-support-vector point -- boundary should be identical
non_support_idx = [i for i in range(n_total) if i not in svm.support_][0]
X_reduced = np.delete(X, non_support_idx, axis=0)
y_reduced = np.delete(y_labels, non_support_idx)
svm_reduced = SVC(kernel="linear", C=1000).fit(X_reduced, y_reduced)
print(f"\nAfter removing a non-support point, w={np.round(svm_reduced.coef_[0], 3)}, "
      f"b={svm_reduced.intercept_[0]:.3f}")
print(f"Boundary unchanged: {np.allclose(svm.coef_, svm_reduced.coef_, atol=1e-3)}")

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(X[:, 0], X[:, 1], c=y_labels, cmap="coolwarm", edgecolors="k", s=50, label="training points")
ax.scatter(svm.support_vectors_[:, 0], svm.support_vectors_[:, 1], s=200, facecolors="none",
           edgecolors="black", linewidths=2, label="support vectors (active constraint)")
xx = np.linspace(X[:, 0].min() - 1, X[:, 0].max() + 1, 100)
yy = -(svm.coef_[0, 0] * xx + svm.intercept_[0]) / svm.coef_[0, 1]
ax.plot(xx, yy, 'k--', label="decision boundary")
ax.set_title("Complementary slackness in action: only circled points\n(active constraints) determine the boundary")
ax.legend(fontsize=9)
plt.show()

## 🏋️ Exercises

Six levels, in order: mechanical computation → conceptual understanding → derivation →
implementation → application → open-ended challenge. Solutions/outlines are in the collapsed
`Solution` blocks below each exercise — try before revealing.

### 🔢 COMPUTE
1. Using Lagrange multipliers, minimize $f(x,y)=x^2+4y^2$ subject to $x+y=3$ by hand. Verify
   with `sp.solve` on the stationarity equations, and cross-check with
   `scipy.optimize.minimize`. As a sanity check, confirm $\nabla f$ and $\lambda\nabla g$ are
   exactly equal at your solution, and confirm the constraint $x+y=3$ is exactly satisfied.

<details>
<summary>💡 Solution</summary>

$\mathcal{L}=x^2+4y^2-\lambda(x+y-3)$. Stationarity: $2x=\lambda$, $8y=\lambda$, $x+y=3$. From
the first two, $x=\lambda/2$ and $y=\lambda/8$; substituting into the constraint,
$\lambda/2+\lambda/8=3\Rightarrow5\lambda/8=3\Rightarrow\lambda=24/5$. So $x=12/5=2.4$,
$y=3/5=0.6$ — matching both `sp.solve` and `scipy.optimize.minimize` exactly. Sanity check:
$\nabla f=(2x,8y)=(4.8,4.8)$ and $\lambda\nabla g=\frac{24}{5}(1,1)=(4.8,4.8)$ — identical, as
KKT/Lagrange stationarity requires; and $x+y=2.4+0.6=3$, satisfying the constraint exactly.

</details>

### 💭 UNDERSTAND
2. For the inequality-constrained problem in the Implementation cell, change the objective to
   $(x-0.5)^2+(y-0.5)^2$ (so the unconstrained optimum $(0.5,0.5)$ satisfies $x+y\le2$ with room
   to spare, unlike $(2,2)$ which violates it). Predict, without solving anything, what the KKT
   multiplier $\mu$ will be at the true constrained optimum, then verify by solving the KKT
   system directly. Explain your prediction using complementary slackness.

<details>
<summary>💡 Solution</summary>

$\mu=0$ exactly — verified: the unconstrained stationary point is $(0.5,0.5)$, `scipy.optimize`
confirms this is also the constrained optimum, and $x+y=1<2$, strictly inside the feasible
region (unlike the earlier boundary case $(1,1)$ where $x+y$ equals exactly $2$). Complementary
slackness states $\mu\, g(\mathbf{w}^*)=0$: since $g(\mathbf{w}^*)=1-2=-1\ne0$ here, the equation
forces $\mu=0$, not the other way around — the constraint being *strictly* satisfied is what
determines the multiplier is zero, not a separate assumption. Intuitively, the constraint
$x+y\le2$ never "pushes back" on this optimization at all: the unconstrained minimum already
lives well inside the feasible region, so adding the constraint changes nothing about where the
optimum sits, and KKT reports that by zeroing out the multiplier that would otherwise measure the
constraint's effect.

</details>

### ✏️ DERIVE
3. The Lagrange multiplier $\lambda$ has a sensitivity interpretation: for
   $p^*(c)=\min\{f(\mathbf{w}) : g(\mathbf{w})=c\}$, the envelope theorem gives
   $dp^*/dc = \lambda(c)$ at the corresponding optimum. Derive this for
   $f(x,y)=x^2+4y^2$ subject to $x+y=c$: solve the Lagrangian system symbolically as a function
   of $c$ to get $p^*(c)$ and $\lambda(c)$ separately, differentiate $p^*(c)$, and confirm the
   two match. Verify numerically at $c=3$ using a finite difference of `scipy.optimize.minimize`
   run at $c=3\pm h$ for small $h$.

<details>
<summary>💡 Solution outline</summary>

Repeating exercise 1's stationarity with $c$ symbolic: $x=\lambda/2$, $y=\lambda/8$,
$x+y=c\Rightarrow\lambda=8c/5$, giving $x=4c/5$, $y=c/5$. Substituting back:
$p^*(c)=x^2+4y^2=(4c/5)^2+4(c/5)^2=16c^2/25+4c^2/25=20c^2/25=4c^2/5$. Differentiating,
$dp^*/dc=8c/5$ — exactly $\lambda(c)=8c/5$ derived independently above, confirming the envelope
theorem in this case. At $c=3$: $\lambda(3)=8(3)/5=4.8$, matching exercise 1's $\lambda=24/5=4.8$
exactly. Numerically, a central finite difference
$\big(p^*(3+h)-p^*(3-h)\big)/(2h)$ using `scipy.optimize.minimize` at $h=10^{-5}$ gives
$4.800000$, matching the symbolic $\lambda(3)=4.8$ to at least 4 decimal places — confirming that
the multiplier isn't just an algebraic byproduct of solving the system, it is *literally* the
rate at which the optimal value would change if the constraint were relaxed or tightened, exactly
as the Theory section claims.

</details>

### 🐍 IMPLEMENT
4. The Theory section's penalty method converts $\min f(\mathbf{w})$ s.t. $g(\mathbf{w})\le0$
   into the unconstrained problem $\min f(\mathbf{w})+\rho\max(0,g(\mathbf{w}))^2$. Implement
   gradient descent directly on this penalized objective for the notebook's own
   inequality-constrained problem ($f=(x-2)^2+(y-2)^2$, $g=x+y-2$), for growing
   $\rho\in\{1,10,100,1000,10000\}$, and check the penalized solution's distance to the true KKT
   solution $(1,1)$ found symbolically in the Implementation cell.

<details>
<summary>✅ How to know you're right</summary>

The distance to the true KKT solution should shrink roughly proportionally to $1/\rho$ as $\rho$
grows (verified: at $\rho=10,100,1000,10000$ the residual constraint violation
$g(\mathbf{w}_\rho)$ comes out to $\approx0.0952, 0.00995, 0.000999, 0.0001$ — each within a
percent or two of the closed-form prediction $\mu_{\text{true}}/(2\rho)$ with
$\mu_{\text{true}}=2$, the KKT multiplier found symbolically for this exact problem). The one
thing you must handle carefully: as $\rho$ grows, the penalized objective's effective curvature
near the boundary grows too (roughly $2+2\rho$), so gradient descent's *own* stability threshold
$\eta<2/L$ from `04.02`/`04.03` shrinks in lockstep — using a fixed learning rate that worked
for small $\rho$ will diverge or stall at large $\rho$ (verified directly: a learning rate that
converges cleanly at $\rho=100$ produces garbage at $\rho=100{,}000$ unless correspondingly
reduced). This is the practical downside the Theory section alludes to when it calls the penalty
method "crude" — large $\rho$ buys constraint accuracy but at the cost of an increasingly
ill-conditioned unconstrained problem.

</details>

### 🤖 APPLY
5. Real SVMs almost never use the hard-margin formulation shown in the AI section — they use a
   **soft-margin** SVM, `sklearn`'s hyperparameter `C`, which allows some margin violation in
   exchange for a wider margin (a form of the Theory section's own penalty-method idea: `C`
   plays the role of $\rho$, penalizing violations rather than forbidding them outright). Using
   the notebook's own overlapping-classes setup, sweep `C` over
   `[0.01, 0.1, 1, 10, 100, 1000]` and track the number of support vectors and the margin width
   ($2/\|\mathbf{w}\|$) at each value.

<details>
<summary>✅ What you should observe</summary>

As `C` grows, the margin should shrink monotonically and the number of support vectors should
generally shrink or stabilize — in one run, margin width goes from about $4.53$ at $C=0.01$ down
to about $1.68$ by $C=10$ and beyond, while support vector count drops from $46$ (out of $60$
points) at $C=0.01$ to a stable $20$ from $C\ge10$ onward. Beyond a certain `C` (here, around
$10$), both quantities stop changing — the soft-margin solution has essentially converged to the
hard-margin solution, since a large enough penalty on violations makes them as good as forbidden,
mirroring exactly how the penalty method's $\rho\to\infty$ limit recovers the constrained
optimum in exercise 4. Small `C` deliberately tolerates more violated margins in exchange for a
wider, more conservative margin — the same "trade objective purity for a softer constraint"
tradeoff the Theory section describes generically, now visible in a specific, widely-used model.

</details>

### 🐍 IMPLEMENT (projected GD)
7. Implement projected gradient descent with **box constraints** from scratch: minimize
   $f(\mathbf{w}) = \|A\mathbf{w} - \mathbf{b}\|^2$ subject to $0 \le w_i \le 1$ for all $i$.
   Generate a random $A \in \mathbb{R}^{20 \times 10}$ and $\mathbf{b} \in \mathbb{R}^{20}$.
   The box projection is simply `np.clip(w, 0, 1)`. Run for 5000 steps and compare your solution
   to `scipy.optimize.minimize` with `bounds=[(0,1)]*10`.

<details>
<summary>✅ How to know you're right</summary>

Your projected GD solution should match scipy's to within $10^{-3}$ in every coordinate. The
convergence curve should show $O(1/T)$ sublinear decrease on a log-log plot (slope $\approx -1$).
Some coordinates of $\mathbf{w}^*$ should sit exactly at 0 or 1 — these are the active box
constraints, analogous to the active inequality constraints in the KKT section. Verify that the
gradient at $\mathbf{w}^*$ points *into* the box at every active boundary (the projected-GD
analogue of dual feasibility: the constraint "pushes back" against the gradient).

</details>

### 🤖 APPLY (SVM dual via projected GD)
8. The SVM dual problem is: maximize $\sum_i \alpha_i - \frac{1}{2}\sum_{i,j} \alpha_i \alpha_j
   y_i y_j (\mathbf{x}_i \cdot \mathbf{x}_j)$ subject to $0 \le \alpha_i \le C$ and
   $\sum_i \alpha_i y_i = 0$. Rewrite this as *minimizing* the negative dual, and solve it using
   projected GD: the constraint set is the intersection of a box ($0 \le \alpha_i \le C$) and a
   hyperplane ($\boldsymbol{\alpha}^T \mathbf{y} = 0$). Use alternating projection (project onto
   the box, then project onto the hyperplane, repeat) as an approximate feasibility step. Test on
   the notebook's toy dataset and compare your $\alpha_i$ values to `svm.dual_coef_`.

<details>
<summary>✅ How to know you're right</summary>

Your recovered support vectors (indices where $\alpha_i > \epsilon$) should match
`svm.support_` exactly. The dual objective at convergence should match
$-\frac{1}{2}\|\mathbf{w}\|^2 + \sum \alpha_i$ computed from sklearn's solution to within a few
percent. The hyperplane $\mathbf{w} = \sum \alpha_i y_i \mathbf{x}_i$ recovered from your
$\alpha$ should be parallel to `svm.coef_` (cosine similarity $> 0.99$). This exercise connects
the dual-problem theory from the Theory section to the projected-GD algorithm, solving a real
constrained optimization problem entirely via first-order feasible-set methods.

</details>

### 🚀 CHALLENGE
6. Increase the overlap between the two classes in the AI section (reduce the distance between
   the cluster centers, e.g. progressively from $(-2,-2)$/$(2,2)$ down toward
   $(-0.25,-0.25)$/$(0.25,0.25)$) and re-fit the SVM at each overlap level. Characterize how the
   number of support vectors and the training accuracy change as separation shrinks, and
   investigate what happens to the dual coefficients `svm.dual_coef_` themselves (not just how
   many are active) as classes start to genuinely overlap rather than merely sit close together.

<details>
<summary>🔍 What a strong answer covers</summary>

A strong answer confirms the count of active constraints (support vectors) rises sharply and
consistently as the classes move closer — in one sweep, from $2$ support vectors (out of 40
points) at cluster distance $2.0$, to $8$ at distance $0.5$, to $20$ at distance $0.25$ — robust
across multiple random seeds (the "far" setting consistently gives 2-3 support vectors, the
"close" setting consistently gives 8-14). Intuitively: with well-separated clusters, only the
few points nearest the boundary are ever at risk of a tight margin, so complementary slackness
zeroes out almost every multiplier; as clusters overlap, more and more points sit close enough to
the boundary (or beyond it) that the margin constraint binds for them too. A strong answer digs
further into `dual_coef_` itself and notices a second, sharper phenomenon: with the notebook's
$C=1000$, well-separated data keeps every $|\alpha_i|$ far below $C$ (comfortably around $0.13$
in one run) — every active constraint is merely *touching* the margin. But once classes overlap
enough, several $|\alpha_i|$ hit the ceiling $C$ exactly — these are points whose margin
constraint isn't just active but genuinely *violated*, and $C$ (not the KKT multiplier itself) is
what caps how much any single point's violation can cost. This distinguishes two different
reasons a constraint can be "active" — right at the boundary vs. pushed past it — that the
notebook's hard-margin framing doesn't have room to show, and connects to why soft-margin SVMs
report both a support-vector count and, separately, how many of those are at the $C$ bound.

</details>

---

## 📚 Further Reading
- Boyd & Vandenberghe, *Convex Optimization*, Ch. 5 (Duality — Sec. 5.2.3 has the
- Duchi et al., ["Efficient Projections onto the $\ell_1$-Ball"](https://stanford.edu/~jduchi/projects/DuchiShSiCh08.pdf) (2008)
  zero-strictly-feasible-point example verified numerically above)
- Cortes & Vapnik, ["Support-Vector Networks"](https://link.springer.com/article/10.1007/BF00994018)

---

*Next notebook: [Learning Rate Schedules](06_learning_rate_schedules.ipynb)*